# Laboratorio de regresión - 4

|                |   |
:----------------|---|
| **Nombre**     | Cecilia Quiroz Hidalgo |
| **Fecha**      | 14/09/2026 |
| **Expediente** | 757942  |

## Modelos penalizados

Hasta ahora la función de costo que usamos para decidir qué tan bueno es nuestro modelo al momento de ajustar es:

$$ \text{RSS} = \sum_{i=1}^n e_i^2 = \sum_{i=1}^n (y_i - \hat{y_i})^2 $$

Dado que los errores obtenidos son una combinación de sesgo y varianza, puede ser que se sesgue un parámetro para minimizar el error. Esto significa que el modelo puede decidir que la salida no sea una combinación de los factores, sino una fuerte predilección sobre uno de los factores solamente. 

E.g. se quiere ajustar un modelo

$$ \hat{z} = \hat{\beta_0} + \hat{\beta_1} x + \hat{\beta_2} y $$

Se ajusta el modelo y se decide que la mejor decisión es $\hat{\beta_1} = 10000$ y $\hat{\beta_2}=50$. Considera limitaciones de problemas reales:
- Quizás los parámetros son ajustes de maquinaria que se deben realizar para conseguir el mejor producto posible, y que $10000$ sea imposible de asignar.
- Quizás los datos actuales están sesgados y sólo hacen parecer que uno de los factores importa más que el otro.

Una de las formas en las que se puede mitigar este problema es penalizando a los parámetros del modelo, cambiando la función de costo:

$$ \text{RSS}_{L2} = \sum_{i=1}^n e_i^2  + \lambda \sum_{j=1}^p \hat{\beta_j}^2 $$

El *L2* significa que se está agregando una penalización de segundo orden. Lo que hace esta penalización es que los factores ahora sólo tendrán permitido crecer si hay una reducción al menos proporcional en el error (sacrificamos sesgo, pero reducimos la varianza).

Asimismo, existe la penalización *L1*

$$ \text{RSS}_{L1} = \sum_{i=1}^n e_i^2  + \lambda \sum_{j=1}^p |\hat{\beta_j}| $$

A las penalizaciones *L2* y *L1* se les conoce también como Ridge y Lasso, respectivamente.

Para realizar una regresión con penalización de Ridge o de Lasso usamos el objeto `Ridge(alpha=?)` o `Lasso(alpha=?)` en lugar de `LinearRegression()` de `sklearn`.

Utiliza el dataset de publicidad (Advertising.csv), utiliza train-test-split de 70/30 y realiza 3 regresiones múltiples:

$$ \text{sales} = \beta_0 + \beta_1 (\text{TV}) + \beta_2 (\text{radio}) + \beta_3 (\text{newspaper}) + \epsilon $$

1. Sin penalización
2. Con penalización L2
3. Con penalización L1

¿Qué puedes observar al ajustar los valores de `alpha`? 

Compara los resultados de los coeficientes utilizando valores diferentes de $\alpha$ y los $R^2$ resultantes.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score


In [2]:
df = pd.read_csv('Advertising.csv')
df.head()

,Unnamed: 0,TV,radio,newspaper,sales
0,1,230.1,37.8,69.2,22.1
1,2,44.5,39.3,45.1,10.4
2,3,17.2,45.9,69.3,9.3
3,4,151.5,41.3,58.5,18.5
4,5,180.8,10.8,58.4,12.9


### Separación de variables y train-test split (70/30)

In [4]:
train_test_split?

Signature:
train_test_split(
    *arrays,
    test_size=None,
    train_size=None,
    random_state=None,
    shuffle=True,
    stratify=None,
)
Docstring:
Split arrays or matrices into random train and test subsets.

Quick utility that wraps input validation,
``next(ShuffleSplit().split(X, y))``, and application to input data
into a single call for splitting (and optionally subsampling) data into a
one-liner.

Read more in the :ref:`User Guide <cross_validation>`.

Parameters
----------
*arrays : sequence of indexables with same length / shape[0]
    Allowed inputs are lists, numpy arrays, scipy-sparse
    matrices or pandas dataframes.

test_size : float or int, default=None
    If float, should be between 0.0 and 1.0 and represent the proportion
    of the dataset to include in the test split. If int, represents the
    absolute number of test samples. If None, the value is set to the
    complement of the train size. If ``train_size`` is also None, it will
    be set to 0.25.

trai

In [5]:
X = df[['TV', 'radio', 'newspaper']]
y = df['sales']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

### Regresión sin penalización (LinearRegression)

In [8]:
msp = LinearRegression()
msp.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [9]:
y_pred_sin = msp.predict(X_test)
r2_sin = r2_score(y_test, y_pred_sin)

In [10]:
msp.coef_

array([ 0.04644207,  0.19122526, -0.00169167])

In [11]:
msp.intercept_

np.float64(2.8244295896521)

In [12]:
r2_sin

0.8693413827230663

### Regresión con penalización L2 (Ridge)

In [13]:
mr = Ridge(alpha=1.0)
mr.fit(X_train, y_train)

,alpha,1.0
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,None


In [14]:
y_pred_ridge = mr.predict(X_test)
r2_ridge = r2_score(y_test, y_pred_ridge)

In [15]:
mr.coef_

array([ 0.04644203,  0.19121846, -0.0016898 ])

In [16]:
mr.intercept_

np.float64(2.8245364229101675)

In [17]:
r2_ridge

0.8693422888407416

### Regresión con penalización L1 (Lasso)

In [18]:
ml = Lasso(alpha=1.0)
ml.fit(X_train, y_train)

,alpha,1.0
,fit_intercept,True
,precompute,False
,copy_X,True
,max_iter,1000
,tol,0.0001
,warm_start,False
,positive,False
,random_state,None
,selection,'cyclic'


In [19]:
y_pred_lasso = ml.predict(X_test)
r2_lasso = r2_score(y_test, y_pred_lasso)

In [20]:
ml.coef_

array([ 0.0462995 ,  0.18609045, -0.        ])

In [21]:
ml.intercept_

np.float64(2.9128649634493673)

In [22]:
r2_lasso

0.8699793457267788

### Comparación de coeficientes y R^2 con distintos valores de alpha

In [23]:
alphas = [0.01, 0.1, 1, 10, 100, 1000]

In [24]:
r2_ridge_lista = []

for a in alphas:
    ridge = Ridge(alpha=a)
    ridge.fit(X_train, y_train)
    r2 = r2_score(y_test, ridge.predict(X_test))
    r2_ridge_lista.append(r2)
    print('alpha =', a, '| coeficientes:', ridge.coef_, '| R^2:', r2)

alpha = 0.01 | coeficientes: [ 0.04644207  0.1912252  -0.00169166] | R^2: 0.869341391787323
alpha = 0.1 | coeficientes: [ 0.04644206  0.19122458 -0.00169149] | R^2: 0.8693414733628333
alpha = 1 | coeficientes: [ 0.04644203  0.19121846 -0.0016898 ] | R^2: 0.8693422888407416
alpha = 10 | coeficientes: [ 0.04644166  0.19115727 -0.00167292] | R^2: 0.8693504159208026
alpha = 100 | coeficientes: [ 0.04643796  0.19054768 -0.00150501] | R^2: 0.8694289371943895
alpha = 1000 | coeficientes: [4.64011323e-02 1.84673286e-01 8.96927225e-05] | R^2: 0.8699584368625939


In [25]:
r2_lasso_lista = []

for a in alphas:
    lasso = Lasso(alpha=a)
    lasso.fit(X_train, y_train)
    r2 = r2_score(y_test, lasso.predict(X_test))
    r2_lasso_lista.append(r2)
    print('alpha =', a, '| coeficientes:', lasso.coef_, '| R^2:', r2)

alpha = 0.01 | coeficientes: [ 0.04644044  0.19116086 -0.00165234] | R^2: 0.8693511310686146
alpha = 0.1 | coeficientes: [ 0.04642566  0.1905936  -0.00130166] | R^2: 0.8694340398230657
alpha = 1 | coeficientes: [ 0.0462995   0.18609045 -0.        ] | R^2: 0.8699793457267788
alpha = 10 | coeficientes: [0.04516509 0.14798052 0.        ] | R^2: 0.8639228904634528
alpha = 100 | coeficientes: [0.03280874 0.         0.        ] | R^2: 0.5981805494051196
alpha = 1000 | coeficientes: [0. 0. 0.] | R^2: -0.00044423209597876934


In [26]:
tabla_resultados = pd.DataFrame({
    'alpha': alphas,
    'r2_ridge': r2_ridge_lista,
    'r2_lasso': r2_lasso_lista
})
tabla_resultados

,alpha,r2_ridge,r2_lasso
0,0.01,0.869341,0.869351
1,0.10,0.869341,0.869434
2,1.00,0.869342,0.869979
3,10.00,0.869350,0.863923
4,100.00,0.869429,0.598181
5,1000.00,0.869958,-0.000444


Con alpha pequeño, tanto Ridge como Lasso se comportan casi igual que la regresión sin penalización, los coeficientes y el R^2 son casi los mismos. Conforme alpha crece, ambas penalizaciones reducen los coeficientes hacia cero. La diferencia es que Ridge (L2) los reduce sin llegar exactamente a cero, mientras que Lasso (L1) lleva coeficientes a cero, eliminando variables del modelo. Si alpha se hace demasiado grande, todos los coeficientes tienden a cero y el R^2 empieza a bajar, mostrando que hay un punto óptimo intermedio de alpha que equilibra sesgo y varianza.